# LangChain — Aula 02 · acompanhamento

**Origem:** `Aula02/video01|02|03/main.py` · **Kernel:** o `.venv` **deste módulo**.

## O contrato

- **Eu escrevi:** o encanamento, os modelos falsos e a migração do agente para a API atual.
- **Você escreve:** o porquê, nos blocos **Sua vez**.

## Tudo roda sem chave de API — e aqui isso é vantagem dupla

O video03 original precisa de **duas** chaves (OpenAI e Google) só para demonstrar que uma falha e a outra assume. Com modelo falso, a falha é provocada de propósito e o fallback aparece inteiro, de graça e sempre igual.

Três camadas, como na aula 01: sem modelo (real), chain com falso (encanamento real, texto roteirizado), agente com falso (grafo e ferramenta reais, decisão roteirizada).

## Três adaptações minhas

**1. `with_structured_output` não existe nos modelos falsos** — é a implementação base, que levanta `NotImplementedError`. Fiz um `ClassificadorFalso` que decide por palavra-chave. **Atenção:** uma regra de palavra-chave *não* é o que um LLM faz, e a diferença é justamente a pergunta 2 do bloco 2.

**2. `AgentExecutor` e `create_tool_calling_agent` seguem removidos** no LangChain 1.x, como na aula 01. Migrado para `create_agent`, com o original mostrado ao lado.

**3. O `timeout=0.001` do video03** era o truque para forçar o OpenAI a falhar. Troquei por um modelo falso que levanta `TimeoutError` — mesmo efeito, sem chave e sem depender da rede estar lenta.

Para voltar ao real quando tiver a `GOOGLE_API_KEY`: trocar o modelo falso pelo `ChatGoogleGenerativeAI` em cada célula. Está indicado.

In [ ]:
# [meu] Os três modelos falsos desta aula. Não fazem parte do curso.
from langchain_core.language_models import FakeListChatModel, FakeMessagesListChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableLambda


class ModeloFalso(FakeMessagesListChatModel):
    """Devolve mensagens roteirizadas. `bind_tools` é exigido pelo create_agent."""

    def bind_tools(self, tools, **kwargs):
        return self


class ModeloQueFalha(FakeMessagesListChatModel):
    """Levanta TimeoutError sempre — simula o `timeout=0.001` do video03."""

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, *args, **kwargs):
        raise TimeoutError("timeout de 0.001s estourado")


class ClassificadorFalso(FakeListChatModel):
    """Implementa with_structured_output, que os fakes não têm.

    Decide por palavra-chave. Um LLM decidiria por significado — ver bloco 2.
    """

    def with_structured_output(self, schema, **kwargs):
        def decidir(entrada):
            texto = str(entrada).lower()
            financeiro = ("fatura", "estorno", "pagamento", "cobran", "boleto")
            return schema(setor="financeiro" if any(k in texto for k in financeiro) else "tecnico")
        return RunnableLambda(decidir)


print("modelos falsos prontos — nenhuma chamada de rede neste notebook.")

---
# Bloco 1 · video01 — LCEL de verdade

## A chain de duas passadas

A aula 01 tinha três elos. Esta tem sete, e passa **duas vezes** pelo modelo: a primeira gera um texto, a segunda revisa o texto gerado.

O elo que merece atenção é `{"texto_gerado": RunnablePassthrough()}` — um **dicionário no meio do pipe**. Ele existe porque o segundo prompt espera uma variável chamada `texto_gerado`, mas o que chega ali é uma string solta.

```python
# original: model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)   # original
model = FakeListChatModel(responses=[                                             # [meu]
    "LCEL eh uma linguagem declarativa pra compor componentes com o operador pipe.",
    "LCEL é uma linguagem declarativa para composição de componentes por meio do operador pipe.",
])
parser = StrOutputParser()

prompt_gerador = ChatPromptTemplate.from_template(
    "Gere uma breve descrição técnica sobre o conceito de: {topico}"
)
prompt_revisor = ChatPromptTemplate.from_template(
    "Melhore o seguinte texto técnico, corrigindo eventuais termos e tornando-o mais formal:\n\n{texto_gerado}"
)

def limpar_texto(texto: str) -> str:
    # Simula uma limpeza ou remoção de caracteres indesejados
    return texto.strip().replace("  ", " ")

runnable_limpeza = RunnableLambda(limpar_texto)

chain = (
    prompt_gerador
    | model
    | parser
    | runnable_limpeza
    | {"texto_gerado": RunnablePassthrough()}  # Mantém o dado vivo no dicionário para o próximo prompt
    | prompt_revisor
    | model
    | parser
)

print("elos da chain:")
for i, passo in enumerate(chain.steps, 1):
    print(f'  {i}. {type(passo).__name__}')

In [ ]:
print("=== 1. DEMONSTRANDO .invoke() ===")
resultado_invoke = chain.invoke({"topico": "LangChain Expression Language (LCEL)"})
print(resultado_invoke)

In [ ]:
print("=== 2. DEMONSTRANDO .stream() ===")
# Excelente para mostrar o surgimento de tokens em tempo real na gravação
pedacos = []
for chunk in chain.stream({"topico": "RunnableParallel"}):
    print(chunk, end="", flush=True)
    pedacos.append(chunk)
print()
print(f'\n[chegaram {len(pedacos)} pedaços] primeiros 8: {pedacos[:8]}')

In [ ]:
print("=== 3. DEMONSTRANDO .batch() ===")
# Execução paralela nativa de múltiplos inputs
inputs = [
    {"topico": "Large Language Models"},
    {"topico": "API Idempotency"},
]
resultados_batch = chain.batch(inputs)
for i, res in enumerate(resultados_batch):
    print(f"Resultado {i+1}:\n{res}\n")

### Sua vez

1. Os três métodos rodaram a **mesma** chain. Em uma frase cada: o que `.invoke()`, `.stream()` e `.batch()` fazem de diferente, e em que situação de produto você usaria cada um.
2. O `.stream()` entregou vários pedaços. Olhe o tamanho deles e pense no caso real: numa chain de **duas** passadas pelo modelo, o usuário consegue ver o texto surgindo desde o primeiro token? Justifique a partir da ordem dos elos.
3. Tire o `{"texto_gerado": RunnablePassthrough()}` da chain e rode. O erro que aparece explica para que serve aquele dicionário — descreva em uma frase o que ele faz com o dado que passa por ele.

_

_

---
# Bloco 2 · video02 — Roteamento por intenção

## O schema que obriga o modelo a responder estruturado

`with_structured_output(RoteadorSuporte)` muda o contrato de saída: em vez de texto livre, o modelo precisa devolver algo que caiba no schema Pydantic. Os dois `field_validator` existem porque o modelo pode responder `"Financeiro"`, `"técnico"`, `"TECNICO"` — e o código quer só `financeiro` ou `tecnico`.

A classe abaixo é a do curso, sem alteração nenhuma.

In [ ]:
from pydantic import BaseModel, Field, field_validator
import unicodedata

class RoteadorSuporte(BaseModel):
    """Classifica a intenção do usuário para direcionar ao setor correto."""
    setor: str = Field(
        description="O setor para onde a dúvida deve ser enviada baseado no contexto."
    )

    @field_validator("setor", mode="before")
    def _normalize_setor(cls, v):
        if isinstance(v, str):
            s = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii")
            s = s.strip().lower()
            if "finance" in s:
                return "financeiro"
            if "tecn" in s or "suporte" in s or "tecnico" in s:
                return "tecnico"
            return s
        return v

    @field_validator("setor")
    def _validate_setor(cls, v):
        if v not in ("financeiro", "tecnico"):
            raise ValueError("setor must be 'financeiro' or 'tecnico'")
        return v

# Os validators rodam sem modelo nenhum — é Pydantic puro:
for entrada in ["Financeiro", "TÉCNICO", "suporte", "marketing"]:
    try:
        print(f'  {entrada:12s} -> {RoteadorSuporte(setor=entrada).setor}')
    except Exception as e:
        print(f'  {entrada:12s} -> REJEITADO: {type(e).__name__}')

In [ ]:
# model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)   # original
model = FakeListChatModel(responses=[                                             # [meu]
    "Verifique o log do PostgreSQL e confirme se o serviço está ouvindo na porta 5432.",
    "Para solicitar o estorno de uma fatura duplicada, abra um chamado no portal financeiro.",
])

# [meu] o original é: llm_classificador = model.with_structured_output(RoteadorSuporte)
llm_classificador = ClassificadorFalso(responses=["x"]).with_structured_output(RoteadorSuporte)

prompt_financeiro = ChatPromptTemplate.from_template(
    "Você é um especialista financeiro. Responda a seguinte dúvida sobre faturamento de forma clara e cordial: {pergunta}"
)
prompt_tecnico = ChatPromptTemplate.from_template(
    "Você é um engenheiro de software sênior. Responda a seguinte dúvida de TI de forma técnica e objetiva: {pergunta}"
)
parser = StrOutputParser()

chain_financeira = prompt_financeiro | model | parser
chain_tecnica = prompt_tecnico | model | parser

def rotear_consulta(dados):
    intencao = dados["intencao"].setor
    pergunta_original = dados["pergunta"]

    print(f"\n[Roteador] -> Intenção detectada: {intencao.upper()}")
    print(f"[Roteador] -> o que chegou em dados['pergunta']: {pergunta_original!r}")

    if intencao == "financeiro":
        return chain_financeira.invoke({"pergunta": pergunta_original})
    elif intencao == "tecnico":
        return chain_tecnica.invoke({"pergunta": pergunta_original})

# O dicionário aqui é um RunnableParallel implícito: os dois ramos rodam sobre a mesma entrada
chain_principal = {
    "intencao": ChatPromptTemplate.from_template("{pergunta}") | llm_classificador,
    "pergunta": RunnablePassthrough(),
} | RunnableLambda(rotear_consulta)

print("=== TESTE 1: INPUT TÉCNICO ===")
print(chain_principal.invoke({"pergunta": "Estou recebendo erro 500 ao tentar conectar no banco PostgreSQL local."}))
print("\n" + "="*40)
print("=== TESTE 2: INPUT FINANCEIRO ===")
print(chain_principal.invoke({"pergunta": "Minha fatura veio duplicada este mês, como solicito o estorno?"}))

### Sua vez

1. Acrescentei um `print` dentro do `rotear_consulta` mostrando o que chega em `dados["pergunta"]`. Olhe esse valor com atenção: é a string da pergunta, ou outra coisa? Rastreie quem colocou aquilo ali, e diga o que o `{pergunta}` do prompt acaba recebendo.
2. O meu `ClassificadorFalso` decide por palavra-chave: procura `fatura`, `estorno`, `cobran`. Escreva **duas** perguntas de usuário — uma financeira e uma técnica — que um LLM classificaria certo e a regra de palavra-chave erraria. Isso define o que `with_structured_output` compra de verdade.
3. O `RoteadorSuporte` tem dois validators: um normaliza antes, outro rejeita depois. Por que não bastaria o segundo? Pense em quem produz o valor que chega ali.
4. O dicionário de `chain_principal` roda os dois ramos sobre a mesma entrada. Um deles chama o modelo, o outro não. Que consequência isso tem em custo e latência, e o que mudaria se o roteamento precisasse de três setores em vez de dois?

_

_

---
# Bloco 3 · video03 — Fallback entre modelos

> ### ⚠️ Mesma migração da aula 01
>
> O original usa `create_tool_calling_agent` + `AgentExecutor`, ambos removidos no LangChain 1.x:
>
> ```python
> agent_primario   = create_tool_calling_agent(llm_primario, tools, prompt)
> agent_secundario = create_tool_calling_agent(llm_secundario, tools, prompt)
> agent_resiliente = agent_primario.with_fallbacks([agent_secundario])
> agent_executor   = AgentExecutor(agent=agent_resiliente, tools=tools, verbose=True)
> ```
>
> Com `create_agent`, o grafo compilado **já é** o executor, então o `AgentExecutor` some e o `.with_fallbacks()` passa a ser aplicado sobre o próprio agente.
>
> O original também força a falha com `ChatOpenAI(..., timeout=0.001, max_retries=0)`. Aqui o `ModeloQueFalha` levanta `TimeoutError` direto — mesmo efeito, sem chave e sem depender da rede.

In [ ]:
from langchain_core.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langchain.agents import create_agent   # [meu] substitui AgentExecutor + create_tool_calling_agent

@tool
def calcular_faturamento_anual(empresa: str, receita_mensal: float) -> str:
    """Calcula a projeção de faturamento anual de uma empresa baseada na receita mensal."""
    # A docstring funciona como a descrição lida pelo agente para decidir o uso da ferramenta
    faturamento_projetado = receita_mensal * 12
    return f"A projeção de faturamento anual para a empresa {empresa} é de R$ {faturamento_projetado:,.2f}"

class LogFallbackHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print("🟡 [TENTATIVA] Chamando modelo...")

    def on_llm_error(self, error, **kwargs):
        print(f"🔴 [FALHA] Modelo falhou com o erro: {type(error).__name__}: {error}")
        print("🔄 [FALLBACK] Redirecionando para o modelo secundário...")

    def on_llm_end(self, response, **kwargs):
        print("🟢 [SUCESSO] LLM respondeu com sucesso!")

tools = [calcular_faturamento_anual]
sistema = ("Você é um assistente financeiro corporativo de elite. "
           "Utilize as ferramentas fornecidas sempre que precisar de dados calculados.")

# llm_primario   = ChatOpenAI(model="gpt-5.4-mini", temperature=0, timeout=0.001, max_retries=0)  # original
# llm_secundario = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)                # original
llm_primario = ModeloQueFalha(responses=[AIMessage(content="nunca chega aqui")])                  # [meu]
llm_secundario = ModeloFalso(responses=[                                                          # [meu]
    AIMessage(content="", tool_calls=[{
        "name": "calcular_faturamento_anual",
        "args": {"empresa": "AlfaTech", "receita_mensal": 25000.0},
        "id": "call_1",
    }]),
    AIMessage(content="A projeção de faturamento anual da AlfaTech é de R$ 300.000,00."),
])

agente_primario = create_agent(llm_primario, tools, system_prompt=sistema)
agente_secundario = create_agent(llm_secundario, tools, system_prompt=sistema)
agente_resiliente = agente_primario.with_fallbacks([agente_secundario])

print("=== EXECUTANDO O AGENTE AUTÔNOMO ===")
consulta = "A empresa AlfaTech fatura 25000 reais por mês. Qual a projeção anual deles?"

resposta = agente_resiliente.invoke(
    {"messages": [HumanMessage(content=consulta)]},
    config={"callbacks": [LogFallbackHandler()]},
)

print("\n=== TRAJETÓRIA COMPLETA ===")
for m in resposta["messages"]:
    chamadas = getattr(m, "tool_calls", None)
    if chamadas:
        print(f'{type(m).__name__:13s} | PEDE FERRAMENTA → {chamadas[0]["name"]}({chamadas[0]["args"]})')
    else:
        print(f'{type(m).__name__:13s} | {m.content}')

### Sua vez

1. Conte quantas vezes o `[TENTATIVA]` apareceu e relacione com a trajetória impressa embaixo. Por que o modelo secundário foi chamado **mais de uma vez** depois de uma única falha do primário?
2. O fallback foi aplicado no agente inteiro, não no modelo. Qual a diferença prática entre `agente_primario.with_fallbacks([agente_secundario])` e aplicar o fallback só no `llm` dentro de um agente só? Pense no que é refeito quando a troca acontece.
3. O `with_fallbacks` dispara em **qualquer** exceção. Num sistema real, falha de rede e resposta inválida do modelo são problemas diferentes. Que risco existe em tratar os dois do mesmo jeito?
4. A ferramenta calculou `25000 × 12`. Se o modelo secundário também falhasse, o usuário receberia o quê? Onde, nesse desenho, está a fronteira entre "o sistema se recuperou" e "o sistema mentiu"?

_

_

---
## Fechamento — responda sem olhar para cima

1. Esta aula mostrou três formas de a chain **decidir** algo: o dicionário que roda ramos em paralelo, o `RunnableLambda` que escolhe uma sub-chain, e o `with_fallbacks` que troca de modelo. Em cada caso, quem toma a decisão — o seu código ou o modelo?
2. `with_structured_output` obriga a saída a caber num schema. O que acontece, num cenário real, quando o modelo insiste em responder algo que não cabe? O código desta aula tem alguma defesa para isso?
3. O que você consegue testar numa chain LangChain **sem nenhuma chave de API**, e o que necessariamente exige o modelo real? Essa resposta define o que dá para cobrir com teste automatizado no seu projeto.

_

_

## Ponte para o Tech Challenge

4. O roteador do bloco 2 escolhe entre dois especialistas pela intenção da pergunta. O seu desafio tem perguntas de tipos diferentes — "quais os principais problemas?" é agregação, "o que dizem sobre entrega?" é filtro por tema. Vale rotear? Decida e justifique.
5. O requisito 4 do seu Tech Challenge é abstenção. O `with_fallbacks` é um mecanismo de resiliência: tenta outra coisa quando algo falha. Abstenção é o oposto — parar quando não há base. Os dois podem conviver no mesmo sistema? Onde cada um entra?

_

_